# 1. TAVANT:

In [0]:
USE CATALOG SQL;
USE SCHEMA interviews;

In [0]:
-- Create the table
CREATE OR REPLACE TABLE sql.interviews.travel (
    START_POINT VARCHAR(50),
    END_POINT   VARCHAR(50),
    DISTANCE    INT
);

-- Insert records
INSERT INTO sql.interviews.travel (START_POINT, END_POINT, DISTANCE) VALUES
('Pune', 'Mumbai', 300),
('Mumbai', 'Pune', 300),
('Delhi', 'Agra', 230),
('Agra', 'Delhi', 230),
('Bangalore', 'Mysore', 150),
('Mysore', 'Bangalore', 150),
('Chennai', 'Pondicherry', 160),
('Pondicherry', 'Chennai', 160),
('Hyderabad', 'Vijayawada', 275),
('Vijayawada', 'Hyderabad', 275),
('Jaipur', 'Udaipur', 400),
('Udaipur', 'Jaipur', 400),
('Pune', 'Nashik', 200),
('Nashik', 'Mumbai', 200);


In [0]:
with travel as (
  select *,least(START_POINT, END_POINT) AS LEAST,
greatest(START_POINT, END_POINT) AS GREATEST from sql.interviews.travel)

select * from travel;

In [0]:
-- select * from sql.interviews.travel t1 join sql.interviews.travel t2 
-- on t1.START_POINT = t2.END_POINT and t1.END_POINT = t2.START_POINT and t1.DISTANCE = t2.DISTANCE;

In [0]:
with travel as (
  select *, least(START_POINT, END_POINT) AS LEAST,
greatest(START_POINT, END_POINT) AS GREATEST from sql.interviews.travel)

select distinct least, greatest, distance from travel;

In [0]:
-- Create the table:
CREATE OR REPLACE TABLE sql.interviews.employee_salary (
    emp_id INT,
    salary_date DATE,
    salary INT
);

INSERT INTO sql.interviews.employee_salary VALUES
(1, '2024-01-15', 5000),
(2, '2024-02-15', 5100),
(3, '2024-03-15', 5100),
(4, '2024-04-15', 5500),
(5, '2024-06-15', 5800),
(6, '2024-03-15', 5100);


In [0]:
select *,row_number() over(order by salary desc) as row_nm,
 rank() OVER(order by salary desc) as rank,
 dense_rank() over(order by salary desc) as dense_rank 
 from sql.interviews.employee_salary;

In [0]:
select *, round(
avg(salary) 
over(partition by emp_id order by salary_date asc
range between interval '2' months preceding and current row)
,2) as moving_averagey
 from sql.interviews.employee_salary;

LeetCode

In [0]:
CREATE OR REPLACE TABLE sql.interviews.logs(
    id int,
    num int
);

INSERT INTO sql.interviews.logs VALUES
(1,1),
(2,1),
(3,1),
(4,2),
(5,1),
(6,2),
(7,2);

In [0]:
with consecutive as (
SELECT id,num, 
lead(num,1) over(order by id asc) as next_num,
lead(num,2) over(order by id asc) as next_next_num
from sql.interviews.logs)
select num as ConsecutiveNums from consecutive where num = next_num and num = next_next_num;

In [0]:
CREATE OR REPLACE TABLE sql.interviews.employee (
    id int,
    name STRING,
    salary int,
    managerid int
);

INSERT INTO sql.interviews.employee VALUES
(1,'Joe',70000,3),
(2,'Henry',80000,4),
(3,'Sam',60000,Null),
(4,'Max',90000,Null);

In [0]:
with cte1 as (
    select E1.id emp_id, E1.name emp_name, E1.salary emp_salary, E1.managerId emp_managerid,
E2.id, E2.name, E2.salary mang_salary, E2.managerId from sql.interviews.employee E1 join sql.interviews.employee E2 on E1.managerid = E2.id)

select emp_name from cte1 where emp_salary > mang_salary;

In [0]:
CREATE OR REPLACE TABLE sql.interviews.seating (
    id int,
    name STRING
);

INSERT INTO sql.interviews.seating VALUES 
(1, 'A'),
(2, 'B'),
(3, 'C'),
(4, 'D'),
(5, 'E')





In [0]:
SELECT 
ID, NAME,
CASE
 WHEN id % 2 == 0 THEN LAG(NAME) OVER (ORDER BY ID ASC)
 WHEN id % 2 != 0 THEN coalesce(LEAD(NAME) OVER (ORDER BY ID ASC),NAME)
 END AS NAME
FROM sql.interviews.seating;

# Vision Board

In [0]:
-- 1. Find Duplicates and delete them:
CREATE OR REPLACE TABLE sql.interviews.duplicates (
    id int,name STRING,dept STRING,salary int
);

INSERT INTO sql.interviews.duplicates VALUES 
(1, 'Danish', 'IT',50000),
(2, 'Raghav', 'HR',60000),
(3, 'Arvind', 'IT',50000),
(4, 'Aditya', 'Finance',80000),
(5, 'Deep', 'IT',90000),
(6, 'Shubham', 'HR',60000),
(7, 'Anu', 'HR',20000);

In [0]:
select *, rank() over(partition by dept order by salary desc) as rnk
 from sql.interviews.duplicates order by dept asc, salary desc;

In [0]:
with dupli as (
    select id, name, dept, salary, _metadata.row_index as row_index, row_number() over (partition by name, dept, salary order by id asc) as row_nm
    from sql.interviews.duplicates
)

delete from sql.interviews.duplicates where id in (select id from dupli where row_nm > 1);

In [0]:
%python
#2. Top 3 Salary per Department:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col
data = [
(1, "Ravi",  90000, 10),
(2, "Anu",   120000, 10),
(3, "Kiran", 120000, 10),
(4, "Meera", 80000, 10), 
(5, "John",  70000, 10),
(6, "Priya", 150000, 20), 
(7, "Arjun", 140000, 20), 
(8, "Divya", 140000, 20),
(9 ,"Sneha", 130009, 20),
(10, "Rahul", 95900, 30),
(11, "Asha", 90900, 30),
(12, "Vikram", 85000, 30)
]
columns = ["emp_id", "emp_name", "salary", "dept_id"]

employees = spark.createDataFrame(data, columns)

# window_spec = Window.partitionBy("dept_id").orderBy(col("salary").desc())
# df_ranked = employees.withColumn("rnk", dense_rank().over(window_spec))
# #Top 3 salaries per department (including ties)
# df_top3 = df_ranked.filter(col("rnk") <= 3)
# df_top3.orderBy("dept_id", col("rnk"), col("salary").desc()).show()

In [0]:
-- employees.createOrReplaceTempView("salary")


In [0]:
CREATE OR REPLACE TABLE sql.interviews.Lag_Lead(
    id int
);

INSERT INTO sql.interviews.Lag_Lead values
    (1),
    (2),
    (3),
    (4),
    (5),
    (6),
    (7)
;

In [0]:
select id,
lead(id,2) over(order by id asc) as lead,
lag(id,4) over(order by id asc) as lag
from sql.interviews.Lag_Lead;

In [0]:
CREATE OR REPLACE TABLE sql.interviews.office(
    id int,
    name STRING,
    date date,
    action string
);

insert into sql.interviews.office values
(1, 'A', '2025-05-12', 'Hire'),
(2, 'B', '2025-05-15', 'Hire'),
(3, 'A', '2025-10-10', 'Terminate'),
(4, 'B', '2025-10-12', 'Terminate'),
(5, 'A', '2025-10-15', 'Hire'),
(6, 'C', '2025-10-15', 'Hire');

select * from sql.interviews.office;

In [0]:
select name, date,
lead(date) over(partition by name order by date asc) lead_dt,
action from sql.interviews.office

In [0]:
select name, date as starting_date, lead_dt as end_date from (
select name, date,
lead(date) over(partition by name order by date asc) lead_dt,
action from sql.interviews.office
) where action = 'Hire';

In [0]:
CREATE OR REPLACE TABLE sql.interviews.project(
    emp_id int,
    project_id STRING,
    start_date date,
    end_date date
);

INSERT INTO sql.interviews.project VALUES
(1, 'P1', '2023-01-10', '2023-03-05'),
(2, 'P2', '2023-02-15', '2023-04-20');

SELECT * FROM sql.interviews.project;

In [0]:
with cte1 as (
select *, 
explode(
    sequence(
        cast(date_trunc('month', start_date) as date),
        cast(date_trunc('month', end_date) as date),
        interval 1 month
    )
) as month_start
 from sql.interviews.project),
cte2 as(
 select *, last_day(month_start),least(end_date,last_day(month_start)),greatest(month_start,start_date),
 date_diff(least(end_date,last_day(month_start)),greatest(month_start,start_date)) +1 as diff from cte1)

 select emp_id, project_id, month_start as month, diff as billabale_days from cte2;